# 01 · Veri hazırlama ve veri denetimi (VS13 – kneeKL224)

Bu notebook şunları yapar:
1. Google Drive'ı bağlar ve proje klasörüne geçer
2. Gerekli ek kütüphaneleri kurar, ortam bilgisini kaydeder
3. `KneeXrayData.zip` arşivini indirir, SHA256 özetini hesaplar, **yalnızca `kneeKL224`** klasörünü oturum diskine açar
4. Resmi train/val/test dosya listelerini oluşturur (bölme değiştirilmez)
5. Veri denetimini çalıştırır: bütünlük, sınıf dağılımı, boyutlar, aykırı görüntüler, pHash yakın-kopya taraması, hasta çakışması

**Çalıştırma:** `Çalışma zamanı → Tümünü çalıştır`. Bu aşama GPU gerektirmez.

## 1. Drive ve proje klasörü

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

## 2. Kurulum ve ortam bilgisi

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0
!mkdir -p outputs/env && pip freeze > outputs/env/pip_freeze.txt

from src.utils import env_info, save_json, set_seed
from src import config as C
set_seed(C.SEED)
info = env_info(); save_json(info, C.OUTPUT_DIR / 'env' / 'env_info.json'); info

## 3. İndirme ve çıkarma

`DATA_URL` değişkenine **YZ_2026_Proje_Veri_Seti_Atamalari.xlsx** dosyasındaki VS13 doğrudan indirme bağlantısını yapıştırın. Mendeley bağlantıları yönlendirme içerdiği için indirme `curl -L` ile yapılır ve kesilirse kaldığı yerden devam eder.

İndirilen arşiv (açılmamış zip) bir kez `Drive/YZProje_cache/` klasörüne de kopyalanır. Sonraki notebook'lar veriyi internetten değil bu kopyadan alır (`data_prep.ensure_data()`); açılmış görüntüler Drive'a konmaz.

⏱ İlk çalıştırmada 7,17 GB indirme + SHA256 + çıkarma yaklaşık 10–20 dakika sürer.

In [ ]:
DATA_URL = ''      # ör. 'https://data.mendeley.com/public-files/datasets/56rmx5bjcr/files/.../file_downloaded'
ZIP_PATH = ''      # ör. '/content/drive/MyDrive/KneeXrayData.zip'

from pathlib import Path
from src import data_prep
if Path(C.EXTRACT_DIR).exists() and any(Path(C.EXTRACT_DIR).rglob('kneeKL224')):
    print('Veri zaten açılmış, yalnızca dosya listesi yenileniyor.')
    data_prep.main(['--skip-extract'])
elif DATA_URL:
    data_prep.main(['--url', DATA_URL])
elif ZIP_PATH:
    data_prep.main(['--zip', ZIP_PATH])
else:
    raise ValueError('DATA_URL veya ZIP_PATH doldurulmalı')

In [ ]:
# Ham arşiv artık gerekmiyorsa disk açmak için silinebilir (SHA256 özeti download_info.json'da kayıtlı)
# !rm -f /content/data/raw/KneeXrayData.zip
!df -h . | tail -1
print(open(C.AUDIT_DIR / 'download_info.json').read())

## 4. Veri denetimi

In [ ]:
from src import audit
audit.main([])

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(C.AUDIT_DIR / 'audit_summary.md', encoding='utf-8').read()))

## 5. Figürler

In [ ]:
for f in sorted(C.FIG_DIR.glob('audit_*.png')):
    print(f.name); display(Image(filename=str(f), width=900))

## 6. Sonraki adım

`outputs/audit/audit_summary.md`, `audit_summary.json` ve `outputs/figures/audit_*.png` dosyalarını değerlendirme için paylaşın. Dosya listeleri `outputs/splits/` altında sabitlendi; sonraki tüm deneyler bu listeleri kullanır.